# 05 — Transfer Learning ACC Arena → Salt&Tar (Advanced, 3 pts)
Reuse the NN trained on **ACC Arena** and **fine-tune** it on a *limited* **Salt&Tar** training set, comparing
against the **same network trained from scratch** on that same limited set.

Since the value of transfer learning depends on **how scarce** target-domain data is, we sweep the size of the
limited train set (number of Salt&Tar users) and evaluate both approaches on one **fixed test set**: the classic
TL curve. Feature schema is identical to notebook 02 (fixed numeric + 6 traffic one-hot + `BEST_X` neighbours
in the `BEST_ENC` encoding), so the ACC-trained weights transfer directly.

In [ ]:
# === Project config — Team 8: Throughput Prediction in a Dense 5G deployment (with Transfer Learning) ===
RANDOM_SEED      = 42
RESAMPLE_SECONDS = 120           # time granularity: THE dataset-size knob (professor's guidance: coarsen the
                                 # time grid rather than subsample users). Raw cadence is ~3s with frequent 4s
                                 # steps, duplicate stamps and gaps up to 7s (ACC) / ~1s (Salt&Tar); keep >= 10
                                 # so the nearest-alignment tolerance stays above the worst raw gap.
N_USERS          = None          # ACC Arena users. None = ALL (~12k), so the X-closest neighbourhoods are the
                                 # real ones; an int (e.g. 1500, seeded random sample) biases the neighbourhoods
                                 # (X closest searched within the sample) and is only for quick debug runs.
N_USERS_SALT     = 3000          # Salt&Tar users: ALL of them (only ~1/3 are ever active; activity is id-biased)
X_VALUES         = [3, 5, 10]    # number of closest users to experiment with. X=1 excluded by design:
                                 # heavy co-location makes a single arbitrary neighbour uninformative.
                                 # X=0 (no neighbour features) IS produced and trained: it is the baseline
                                 # that quantifies what the closest-users features actually add.
ENCODINGS        = ["pos", "agg"]# neighbour-feature encodings: "pos" = ordered per-neighbour columns
                                 # (nb0_*, nb1_*, ...), "agg" = order-invariant aggregates over the X
                                 # neighbours (sum/mean/count) — rationale in notebook 02.
BEST_X           = 3             # X used for the transfer-learning experiment (pick the best from notebook 04)
BEST_ENC         = "pos"         # encoding used for the transfer-learning experiment (pick from notebook 04)
OUTLIER_PCT      = 99.0          # drop samples with throughput above this TRAIN percentile (None = keep all).
                                 # EDA (notebook 01): at full population the top ~1% of active samples carry
                                 # ~86% of the total variance -> MSE/R2 would be about a handful of rare peaks.
ACTIVE_ONLY      = True           # regress only on ACTIVE users; idle/off have throughput ~0 by definition
MIN_TRAFFIC_TYPE = 2             # keep traffic_type >= this (2=const_rate, 3=video, 4=gaming, 5=http)

# --- data location ---
DRIVE_FOLDER_ID  = "1s1YCWyhN_Fv5sQraTVs4Rga-ATiKPRgo"   # shared Drive folder containing the zip
ZIP_NAME         = "L5GHDD_Dataset.zip"
DATA_ROOT        = "data/raw"     # dataset folders live here after unzip (local default)
PROCESSED_DIR    = "data/processed"
RESULTS_DIR      = "results"

import os, numpy as np, random
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: mount Drive and make outputs PERSIST across notebooks (no-op locally) ===
def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR   = "/content/drive/MyDrive/5g_throughput_team8"   # persistent project folder on your Drive
    PROCESSED_DIR = f"{PROJECT_DIR}/processed"                     # 02 writes here, 03/04/05 read from here
    RESULTS_DIR   = f"{PROJECT_DIR}/results"                       # models, metrics.csv, figures
    print("Outputs persist in:", PROJECT_DIR)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: locate and unzip the dataset (no-op locally) ===
if in_colab():
    import glob, zipfile, subprocess
    DATA_ROOT = "/content/L5GHDD_Dataset"
    if not os.path.isdir(DATA_ROOT):
        cands = glob.glob(f"/content/drive/MyDrive/**/{ZIP_NAME}", recursive=True)
        if not cands:                                   # fallback: download the shared folder
            subprocess.run(["pip", "-q", "install", "gdown"], check=False)
            import gdown
            gdown.download_folder(id=DRIVE_FOLDER_ID, output="/content/_dl", quiet=False, use_cookies=False)
            cands = glob.glob(f"/content/_dl/**/{ZIP_NAME}", recursive=True)
        assert cands, f"{ZIP_NAME} not found. Put it in your Drive or share the folder."
        print("Unzipping", cands[0], "...")
        with zipfile.ZipFile(cands[0]) as z:
            z.extractall(DATA_ROOT)
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
# === Data loading helpers (raw "wide" format -> tidy per-user/per-timestamp table) ===
import glob, re, math
import pandas as pd
from scipy.spatial import cKDTree

def find_venue_dir(data_root, venue_key):
    """venue_key in {'acc','salt'}; robust to the zip's internal layout."""
    pat = {"acc": "*ACC*Arena*", "salt": "*Salt*Tar*"}[venue_key]
    hits = [os.path.join(dp, d) for dp, dn, _ in os.walk(data_root)
            for d in dn if glob.fnmatch.fnmatch(d, pat)]
    hits = sorted(set(hits), key=len)
    assert hits, f"venue '{venue_key}' not found under {data_root}"
    return hits[0]

def file_id_range(path):
    """User-id range covered by a CSV chunk, taken from its file name (…_<start>_<end>.csv)."""
    m = re.search(r'_(\d+)_(\d+)\.csv$', path)
    return int(m.group(1)), int(m.group(2))

def metric_files(venue_dir, subdir_glob, file_glob, user_ids=None):
    """CSV chunks of a metric, keeping only files whose user-id range intersects `user_ids`."""
    subs = glob.glob(os.path.join(venue_dir, subdir_glob))
    assert subs, f"no subdir matching {subdir_glob} in {venue_dir}"
    files = sorted(glob.glob(os.path.join(subs[0], file_glob)), key=lambda p: file_id_range(p)[0])
    if user_ids is not None:
        def overlaps(f):
            a, b = file_id_range(f)
            return any(a <= u <= b for u in user_ids)
        files = [f for f in files if overlaps(f)]
    assert files, f"no files matching {file_glob} in {subdir_glob}"
    return files

def all_user_ids(venue_dir):
    """Every user id present in the venue (derived from the throughput file names)."""
    ids = []
    for f in metric_files(venue_dir, "*Throughput*", "*.csv"):
        a, b = file_id_range(f)
        ids.extend(range(a, b + 1))
    return np.array(sorted(ids))

def build_grid(ref_file, resample_seconds):
    t = pd.read_csv(ref_file, usecols=[0]).iloc[:, 0].values.astype(float)
    return np.arange(t[0], t[-1] + 1, resample_seconds)

def _align(df, grid, tol):
    df = df[~df.index.duplicated(keep="first")]   # raw stamps contain duplicates (diff = 0s)
    return df.reindex(grid, method="nearest", tolerance=tol)

def load_metric(files, value_name, grid, tol, user_ids):
    out = []
    for f in files:
        head = list(pd.read_csv(f, nrows=0).columns)
        cmap = {c: int(re.search(r'(\d+)', c).group(1)) for c in head[1:]}
        use = [head[0]] + [c for c in head[1:] if cmap[c] in user_ids]   # parse only sampled users
        df = pd.read_csv(f, usecols=use).rename(columns={head[0]: "time"}).set_index("time")
        df = df.rename(columns=cmap)
        df = _align(df, grid, tol).astype("float32")   # float32 halves RAM (full population fits Colab)
        out.append(df.reset_index().melt(id_vars="time", var_name="user_id", value_name=value_name))
    return pd.concat(out, ignore_index=True)

def load_positions(files, grid, tol, user_ids):
    """Positions are interleaved blocks of (id, lat, lon, alt, mobileState); mobileState is the
    traffic_type. lat/lon are converted to local metres using ONE shared origin for the whole venue,
    so distances between users coming from different files are consistent."""
    frames = []
    for f in files:
        first = pd.read_csv(f, nrows=1).values.astype(float)
        ids_all = first[0, 1::5].astype(int)
        blocks = [k for k, u in enumerate(ids_all) if u in user_ids]
        if not blocks:
            continue
        use = [0] + [1 + 5 * k + j for k in blocks for j in range(5)]    # parse only sampled users
        df = pd.read_csv(f, usecols=use)
        t = df.iloc[:, 0].values.astype(float)
        arr = df.values.astype(float)
        ids = arr[0, 1::5].astype(int)
        vals = {"lat": arr[:, 2::5], "lon": arr[:, 3::5], "z": arr[:, 4::5], "traffic_type": arr[:, 5::5]}
        m = None
        for name, v in vals.items():
            d = pd.DataFrame(v, index=t, columns=ids)
            d.index.name = "time"
            d = _align(d, grid, tol).reset_index().melt(id_vars="time", var_name="user_id", value_name=name)
            m = d if m is None else m.merge(d, on=["time", "user_id"])
        frames.append(m)
    pos = pd.concat(frames, ignore_index=True)
    lat0, lon0 = pos.lat.mean(), pos.lon.mean()          # single origin for the whole venue
    R = 6371000.0
    pos["x"] = R * np.radians(pos.lon - lon0) * math.cos(math.radians(lat0))
    pos["y"] = R * np.radians(pos.lat - lat0)
    return pos[["time", "user_id", "x", "y", "z", "traffic_type"]]

def assemble(venue_key, n_users, resample_seconds, random_users=False):
    """Return a tidy DataFrame with one row per (user, timestamp).
    n_users=None uses the venue's FULL population (recommended: the X-closest neighbourhoods are the
    real ones). random_users=True draws n_users uniformly from the full population (seeded,
    reproducible); otherwise the first n_users ids are used."""
    venue = find_venue_dir(DATA_ROOT, venue_key)
    pop = all_user_ids(venue)
    if n_users is None:
        user_ids = set(int(u) for u in pop)
        print(f"{venue_key}: using ALL {len(user_ids)} users")
    elif random_users:
        rng = np.random.default_rng(RANDOM_SEED)
        user_ids = set(int(u) for u in rng.choice(pop, size=min(n_users, len(pop)), replace=False))
        print(f"{venue_key}: sampled {len(user_ids)} random users out of {len(pop)}")
    else:
        user_ids = set(range(n_users))
    # nearest-alignment tolerance: half the grid step, but never below the raw data's worst gap.
    # Raw stamps are jittered (ACC: nominal 3s but ~1/3 of steps are 4s, plus duplicates and holes
    # up to 7s), so a floor of 4s guarantees every grid point finds its sample even at fine grids.
    tol = max(resample_seconds / 2, 4.0)
    mf = lambda sub, fg: metric_files(venue, sub, fg, user_ids)
    ref = mf("*Throughput*", "*.csv")[0]
    grid = build_grid(ref, resample_seconds)
    parts = [
        load_metric(mf("*Throughput*",     "*.csv"),    "throughput", grid, tol, user_ids),
        load_metric(mf("*BLER*",           "*.csv"),    "bler",       grid, tol, user_ids),
        load_metric(mf("*PRB*",            "*.csv"),    "prb",        grid, tol, user_ids),
        load_metric(mf("*RU_Association*", "*.csv"),    "ru_id",      grid, tol, user_ids),
        load_metric(mf("*SINR*",           "SINRDL_*.csv"), "sinr_dl", grid, tol, user_ids),
        load_metric(mf("*SINR*",           "SINRUL_*.csv"), "sinr_ul", grid, tol, user_ids),
        load_positions(mf("*Positions*",   "*.csv"), grid, tol, user_ids),
    ]
    df = parts[0]
    for p in parts[1:]:
        df = df.merge(p, on=["time", "user_id"], how="inner")
    df = df.dropna().reset_index(drop=True)
    df["user_id"]      = df["user_id"].astype(int)
    df["traffic_type"] = df["traffic_type"].round().astype(int)
    df["ru_id"]        = df["ru_id"].round().astype(int)
    return df

In [ ]:
# === Closest-users feature engineering (Team-8 specific) ===
# For each (timestamp, user) we append the features of its X nearest users at that
# instant (3-D Euclidean distance on x,y,z). Per neighbour: [dist, sinr_dl, sinr_ul, prb, bler, throughput].
# Vectorised per timestamp — one KDTree query for all users, then numpy gathers (no per-row Python
# loop) — so it stays fast on the FULL ~12k-user population.
NEIGHBOR_FEATS = ["sinr_dl", "sinr_ul", "prb", "bler", "throughput"]

def add_closest_user_features(df, x_max):
    cols = []
    for k in range(x_max):
        cols += [f"nb{k}_dist"] + [f"nb{k}_{c}" for c in NEIGHBOR_FEATS]
    out = np.full((len(df), len(cols)), np.nan, dtype="float32")
    feat = df[NEIGHBOR_FEATS].values
    pos = df[["x", "y", "z"]].values
    for _, idx in df.groupby("time").groups.items():
        idx = np.asarray(idx)
        n = len(idx)
        if n <= 1:
            continue
        pts = pos[idx]
        k = min(x_max + 1, n)
        dist, nb = cKDTree(pts).query(pts, k=k)
        rows = np.arange(n)[:, None]
        not_self = nb != rows                                 # self appears exactly once per row
        order = np.argsort(~not_self, axis=1, kind="stable")  # non-self first, distance order kept
        take = min(x_max, k - 1)
        r = np.repeat(np.arange(n), take)
        c = order[:, :take].ravel()
        block = np.empty((n, take, 1 + len(NEIGHBOR_FEATS)), dtype="float32")
        block[:, :, 0] = dist[r, c].reshape(n, take)
        block[:, :, 1:] = feat[idx[nb[r, c]]].reshape(n, take, -1)
        out[idx, :take * (1 + len(NEIGHBOR_FEATS))] = block.reshape(n, -1)
    return pd.concat([df, pd.DataFrame(out, columns=cols, index=df.index)], axis=1)

def neighbor_cols(x):
    cols = []
    for k in range(x):
        cols += [f"nb{k}_dist"] + [f"nb{k}_{c}" for c in NEIGHBOR_FEATS]
    return cols

# === Order-invariant neighbour aggregates (encoding "agg", same as notebook 02) ===
AGG_FEATS = ["nb_prb_sum", "nb_throughput_sum", "nb_sinr_dl_mean", "nb_sinr_ul_mean",
             "nb_bler_mean", "nb_active_count"]

def aggregate_neighbor_features(frame, x):
    """Aggregates over the X nearest users, computed from the positional nb{k}_* columns."""
    def block(feat):
        return frame[[f"nb{k}_{feat}" for k in range(x)]]
    prb = block("prb")
    return pd.DataFrame({
        "nb_prb_sum":        prb.sum(axis=1, min_count=1),                  # ~ cell load from neighbours
        "nb_throughput_sum": block("throughput").sum(axis=1, min_count=1),
        "nb_sinr_dl_mean":   block("sinr_dl").mean(axis=1),
        "nb_sinr_ul_mean":   block("sinr_ul").mean(axis=1),
        "nb_bler_mean":      block("bler").mean(axis=1),
        "nb_active_count":   (prb.fillna(0) > 0).sum(axis=1).astype(float),
    }, index=frame.index)

In [ ]:
# === Evaluation metrics ===
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
def evaluate(y_true, y_pred):
    return {"MSE": float(mean_squared_error(y_true, y_pred)),
            "MAE": float(mean_absolute_error(y_true, y_pred)),
            "R2":  float(r2_score(y_true, y_pred))}

## Build the Salt&Tar dataset (limited) with the same feature schema

In [ ]:
import json, joblib, numpy as np, pandas as pd
df = assemble("salt", n_users=N_USERS_SALT, resample_seconds=RESAMPLE_SECONDS)
if BEST_X > 0:
    df = add_closest_user_features(df, x_max=BEST_X)
if ACTIVE_ONLY:                                  # same target definition as ACC Arena (notebook 02)
    df = df[df.traffic_type >= MIN_TRAFFIC_TYPE].reset_index(drop=True)
print("Salt&Tar tidy:", df.shape, "| traffic types:", sorted(df.traffic_type.unique()))

BASE_NUM = ["bler","prb","sinr_dl","sinr_ul","x","y","z"]
TRAFFIC_CLASSES = [0,1,2,3,4,5]
SFX = "_agg" if BEST_ENC == "agg" else ""        # file suffix of the chosen encoding (notebooks 02/03)
acc_cols = json.load(open(f"{PROCESSED_DIR}/acc_X{BEST_X}{SFX}_cols.json"))
scaler   = joblib.load(f"{PROCESSED_DIR}/acc_X{BEST_X}{SFX}_scaler.pkl")

def build_like_acc(frame):
    if BEST_ENC == "agg" and BEST_X > 0:
        frame = pd.concat([frame, aggregate_neighbor_features(frame, BEST_X)], axis=1)
        num_cols = BASE_NUM + AGG_FEATS
    else:
        num_cols = BASE_NUM + neighbor_cols(BEST_X)
    num = frame[num_cols].astype("float32"); num = num.fillna(num.median())
    num_s = pd.DataFrame(scaler.transform(num), columns=num_cols, index=frame.index)
    oh = pd.DataFrame({f"traffic_{c}": (frame.traffic_type==c).astype(float) for c in TRAFFIC_CLASSES},
                      index=frame.index)
    X = pd.concat([num_s, oh], axis=1).reindex(columns=acc_cols, fill_value=0.0)
    return X.values.astype("float32")

## Fixed test set + growing "limited" train sets
Half of the Salt&Tar users form a **fixed test set** used for every run. The remaining users form the training
pool; each sweep step takes the first `n` users of the pool. The outlier threshold is computed once on the
training pool (never on test) so the evaluation scope is identical across all sizes.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
users = df.user_id.unique(); rng.shuffle(users)
n_test = len(users)//2
test_u = set(users[:n_test])                    # fixed test set for ALL sizes
pool_u = list(users[n_test:])                   # training pool, sizes draw from its head
TRAIN_SIZES = [5, 10, 25, 50, len(pool_u)]      # Salt&Tar users in the "limited" train set

te   = df[df.user_id.isin(test_u)]
pool = df[df.user_id.isin(set(pool_u))]
if OUTLIER_PCT is not None:
    thr = float(np.percentile(pool.throughput, OUTLIER_PCT))
    pool = pool[pool.throughput <= thr]; te = te[te.throughput <= thr]
    print(f"outlier threshold (p{OUTLIER_PCT} of S&T train pool) = {thr:.2f} Mbps")
Xte, yte = build_like_acc(te), te.throughput.values.astype("float32")
print("fixed test rows:", len(Xte), "| train pool users:", len(pool_u), "| sweep:", TRAIN_SIZES)

## Fine-tuned vs from-scratch at every train size
Same architecture and stopping rule for both; the only differences are the starting weights (ACC-trained vs
random) and the fine-tuning learning rate (lower, to protect the pretrained weights; no layer is frozen
because the target venue's throughput scale differs from the source's).

In [ ]:
import time
from tensorflow import keras

def make_finetune():
    m = keras.models.load_model(f"{RESULTS_DIR}/models/nn_X{BEST_X}{SFX}.keras")
    # all layers trainable: Salt&Tar throughput lives on a different scale than ACC Arena
    # (p99 ~7 vs ~29 Mbps), so freezing layers underfits; the lower LR protects the weights
    m.compile(optimizer=keras.optimizers.Adam(3e-4), loss="mse", metrics=["mae"])
    return m

def make_scratch():
    m = keras.models.clone_model(keras.models.load_model(f"{RESULTS_DIR}/models/nn_X{BEST_X}{SFX}.keras"))
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])  # clone = fresh random weights
    return m

rows = []
for n_u in TRAIN_SIZES:
    tr = pool[pool.user_id.isin(set(pool_u[:n_u]))]
    Xtr, ytr = build_like_acc(tr), tr.throughput.values.astype("float32")
    for setting, make in [("fine-tuned (TL)", make_finetune), ("from scratch", make_scratch)]:
        keras.utils.set_random_seed(RANDOM_SEED)
        m = make()
        es = keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)
        t = time.time()
        m.fit(Xtr, ytr, validation_split=0.2, epochs=60, batch_size=256, callbacks=[es], verbose=0)
        r = evaluate(yte, m.predict(Xte, verbose=0).ravel())
        r.update(setting=setting, train_users=n_u, train_rows=len(Xtr), train_s=round(time.time()-t, 1))
        rows.append(r)
        print(f"users={n_u:>4} rows={len(Xtr):>6} | {setting:16s} R2={r['R2']:.3f} MAE={r['MAE']:.3f} ({r['train_s']}s)")

tl = pd.DataFrame(rows); tl.to_csv(f"{RESULTS_DIR}/transfer_learning.csv", index=False); tl

## Compare — where does transfer learning pay off?

In [ ]:
import matplotlib.pyplot as plt
sizes = sorted(tl.train_users.unique())
xs = list(range(len(sizes)))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for setting, color in [("fine-tuned (TL)", "#2a9d8f"), ("from scratch", "#e76f51")]:
    d = tl[tl.setting == setting].set_index("train_users").loc[sizes]
    ax[0].plot(xs, d.R2,  marker="o", color=color, label=setting)
    ax[1].plot(xs, d.MAE, marker="o", color=color, label=setting)
for a, name in zip(ax, ["R2 (fixed test set)", "MAE (fixed test set, Mbps)"]):
    a.set_xticks(xs); a.set_xticklabels(sizes)
    a.set_xlabel("Salt&Tar users in the limited train set"); a.set_title(name)
    a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/05_transfer_learning.png", dpi=120); plt.show()

**Expected reading.** With very few target-domain users the from-scratch network cannot learn a good mapping and
the ACC-pretrained one dominates; as the limited set grows the two curves converge (and from-scratch may even
edge ahead, since the pretrained weights are tuned to the source venue's throughput scale). The value of transfer learning is the gap in the
low-data regime — report the smallest train size at which each approach becomes usable.